<div style="background:#1F3864;color:white;padding:14px 18px;border-radius:6px"><b>INTERNATIONAL UNIVERSITY – VNU-HCMC · SCHOOL OF COMPUTER SCIENCE AND ENGINEERING</b><br>IT171IU · Statistical Learning</div>

# Lab 4: Resampling Methods: Cross-Validation and the Bootstrap

*Validation-set approach, leave-one-out and k-fold cross-validation, and the bootstrap for standard errors and confidence intervals*

**Chapter:** ISLP Chapter 5 · **Duration:** 150 minutes (3 periods) · **Runs on:** Google Colab (recommended), Jupyter Notebook / JupyterLab, VS Code, PyCharm

**How to use this notebook:** first *File → Save a copy in Drive*. Run the **Setup** cell, then read each text cell and run the code cells in order with **Shift + Enter**. Write your solutions in the cells marked ✍️ and answer the questions in the text cells.

> **Legend:** black = original ISLP lab content (re-formatted) · <span style="color:red">red = content added for IT171IU (theory, worked examples, exercises, setup & submission guide)</span>

In [ ]:
# Student information – fill in and run this cell first
STUDENT_NAME = "Your Full Name"
STUDENT_ID   = "ITDSIUxxxxx"
CLASS_GROUP  = "IT171IU – Lab Group xx"
print(f"Lab 4 | {STUDENT_NAME} | {STUDENT_ID} | {CLASS_GROUP}")

# <span style="color:red">1. Objectives</span>

<span style="color:red">After this lab you will be able to:</span>

- <span style="color:red">Estimate the test error of a model with the **validation-set approach**, **LOOCV** and **k-fold cross-validation**.</span>
- <span style="color:red">Use scikit-learn's `cross_validate()`, `KFold` and `ShuffleSplit`, and ISLP's `sklearn_sm()` wrapper for statsmodels models.</span>
- <span style="color:red">Use cross-validation to **select model flexibility** (e.g. the degree of a polynomial).</span>
- <span style="color:red">Explain the **bias–variance trade-off** between LOOCV and k-fold CV.</span>
- <span style="color:red">Implement the **bootstrap** to estimate the standard error of any statistic and compare it with formula-based standard errors.</span>
- <span style="color:red">Build a **percentile bootstrap confidence interval**.</span>

> 📌 <span style="color:red">**Note** — **Lab plan – 150 minutes (3 periods)**</span>
>
> | <span style="color:red">Time</span> | <span style="color:red">Activity</span> |
> |---|---|
> | <span style="color:red">0 – 20 min</span> | <span style="color:red">Theory review: validation set, LOOCV, k-fold CV, bias–variance of CV, the bootstrap (Section 2)</span> |
> | <span style="color:red">20 – 70 min</span> | <span style="color:red">Guided lab: validation set and CV on Auto; bootstrap for a portfolio statistic and regression coefficients (Section 3)</span> |
> | <span style="color:red">70 – 85 min</span> | <span style="color:red">Extra worked examples: the LOOCV shortcut formula and a percentile bootstrap interval</span> |
> | <span style="color:red">85 – 140 min</span> | <span style="color:red">Exercises 1 – 6 (Section 4); challenge exercises 7 – 8 if time allows</span> |
> | <span style="color:red">140 – 150 min</span> | <span style="color:red">Run all cells, save and submit through the Google Form (Section 5)</span> |

# <span style="color:red">2. Theory</span>

## <span style="color:red">2.1 Why resampling?</span>

<span style="color:red">The training error underestimates the test error, especially for flexible models. Without a large separate test set we can **re-use the training data**: repeatedly fit a model on part of the data and evaluate it on the rest (cross-validation), or repeatedly resample the data to see how much an estimate varies (bootstrap).</span>

## <span style="color:red">2.2 The validation-set approach</span>

<span style="color:red">Randomly split the $n$ observations into a **training set** and a **validation (hold-out) set**; fit on the first, compute the MSE (or error rate) on the second. It is simple, but</span>

1. <span style="color:red">the estimate is **highly variable** – it depends on which observations land in each half;</span>
2. <span style="color:red">only part of the data is used for fitting, so the validation error tends to **overestimate** the test error of the model fitted on all $n$ observations.</span>

## <span style="color:red">2.3 Leave-one-out cross-validation (LOOCV)</span>

<span style="color:red">Fit the model $n$ times, each time leaving out one observation $i$ and computing $\mathrm{MSE}_i = (y_i - \hat y_{(i)})^2$:</span>

$$\color{red}{\mathrm{CV}_{(n)} = \frac{1}{n}\sum_{i=1}^{n}\mathrm{MSE}_i .}$$

<span style="color:red">LOOCV has almost no bias and no randomness, but can be expensive. For least-squares linear or polynomial regression there is a shortcut that needs only **one** fit:</span>

$$\color{red}{\mathrm{CV}_{(n)} = \frac{1}{n}\sum_{i=1}^{n}\left(\frac{y_i-\hat y_i}{1-h_i}\right)^2,}$$

<span style="color:red">where $h_i$ is the leverage of observation $i$.</span>

## <span style="color:red">2.4 k-fold cross-validation</span>

<span style="color:red">Split the data into $k$ folds of (roughly) equal size. Each fold in turn is the validation set and the other $k-1$ folds are used for fitting:</span>

$$\color{red}{\mathrm{CV}_{(k)} = \frac{1}{k}\sum_{j=1}^{k}\mathrm{MSE}_j .}$$

<span style="color:red">With $k = 5$ or $10$ it is much cheaper than LOOCV and usually gives a **more accurate** estimate: LOOCV averages $n$ fitted models that are almost identical (highly correlated), so its estimate has a higher variance. The standard error of the CV estimate is roughly $\mathrm{sd}(\mathrm{MSE}_1,\dots,\mathrm{MSE}_k)/\sqrt{k}$. For classification replace the MSE by the error rate $\frac{1}{n_j}\sum I(y_i\neq\hat y_i)$.</span>

## <span style="color:red">2.5 The bootstrap</span>

<span style="color:red">To estimate the variability of an estimator $\hat\alpha$ (a median, a ratio, a regression coefficient, …):</span>

1. <span style="color:red">draw $B$ **bootstrap samples** of size $n$ from the data **with replacement**;</span>
2. <span style="color:red">compute the estimate $\hat\alpha^{*1},\dots,\hat\alpha^{*B}$ on each sample;</span>
3. <span style="color:red">use their standard deviation as the standard error:</span>

$$\color{red}{\mathrm{SE}_B(\hat\alpha) = \sqrt{\frac{1}{B-1}\sum_{r=1}^{B}\left(\hat\alpha^{*r}-\frac{1}{B}\sum_{r'=1}^{B}\hat\alpha^{*r'}\right)^2}.}$$

<span style="color:red">A simple 95% **percentile interval** is given by the 2.5% and 97.5% quantiles of the $\hat\alpha^{*r}$. Each bootstrap sample contains on average about $1-(1-1/n)^n \approx 63.2\%$ of the distinct original observations.</span>

| <span style="color:red">Method</span> | <span style="color:red">Purpose</span> | <span style="color:red">Typical choice</span> |
|---|---|---|
| <span style="color:red">Validation set</span> | <span style="color:red">quick test-error estimate</span> | <span style="color:red">50/50 or 70/30 split</span> |
| <span style="color:red">LOOCV</span> | <span style="color:red">nearly unbiased test-error estimate</span> | <span style="color:red">small $n$, or least squares (shortcut)</span> |
| <span style="color:red">k-fold CV</span> | <span style="color:red">test error, model selection, tuning</span> | <span style="color:red">$k = 5$ or $10$</span> |
| <span style="color:red">Bootstrap</span> | <span style="color:red">standard errors, confidence intervals</span> | <span style="color:red">$B = 1000$ or more</span> |

# 3. Instructions and Worked Examples

<span style="color:red">Work through this section **in order**: read each explanation, *predict* what a cell will print, then run it. The black text and code are the original ISLP Chapter 5 lab; red notes, checkpoints and extra worked examples were added for IT171IU.</span>

### <span style="color:red">Setup cell</span>

In [ ]:
# [ADDED]
# 0. Setup – run this cell first (installs missing packages and fetches data files).
# On Google Colab the first run takes about 1 minute. If Colab asks you to
# "Restart session", accept, then run this cell once more.
import sys, os, subprocess, importlib.util, urllib.request

def ensure(module, pip_name=None):
    """pip-install a package only if it cannot be imported yet."""
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pip_name or module} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)

ensure("ISLP", "ISLP")

print("Setup complete – Python", sys.version.split()[0])

In [ ]:
# [ADDED] helper for the additional datasets
# Loader for the additional datasets of this course (folder "additional-data/").
# If the CSV file is not found next to the notebook, it is rebuilt from its public source,
# so the notebook also works on Colab without uploading anything.
import pandas as pd
def load_extra(name):
    for folder in ["additional-data", "../additional-data", "/content/additional-data",
                   "/content/drive/MyDrive/additional-data"]:
        path = os.path.join(folder, name + ".csv")
        if os.path.exists(path):
            return pd.read_csv(path)
    from sklearn import datasets as skd
    if name == "diabetes":
        d = skd.load_diabetes(as_frame=True, scaled=False); df = d.frame.rename(columns={"target": "progression"})
    elif name == "breast_cancer":
        d = skd.load_breast_cancer(as_frame=True); df = d.frame.copy()
        df["diagnosis"] = pd.Categorical.from_codes(d.target, ["malignant", "benign"]); df = df.drop(columns="target")
    elif name == "wine":
        d = skd.load_wine(as_frame=True); df = d.frame.rename(columns={"target": "cultivar"})
    elif name == "digits":
        d = skd.load_digits(as_frame=True); df = d.frame.rename(columns={"target": "digit"})
    elif name == "rossi":
        from lifelines.datasets import load_rossi; df = load_rossi()
    else:
        raise ValueError(f"unknown dataset {name!r}")
    return df

In this lab, we explore the resampling techniques covered in this
chapter. Some of the commands in this lab may take a while to run on
your computer.

We again begin by placing most of our imports at this top level.

In [ ]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from sklearn.model_selection import train_test_split

There are several new imports needed for this lab.

In [ ]:
from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

## 3.1 The Validation Set Approach
We explore the use of the validation set approach in order to estimate
the test error rates that result from fitting various linear models on
the  `Auto`  data set.

We use the function `train_test_split()` to split
the data into training and validation sets. As there are 392 observations,
we split into two equal sets of size 196 using the
argument `test_size=196`. It is generally a good idea to set a random seed
when performing operations like this that contain an
element of randomness, so that the results obtained can be reproduced
precisely at a later time. We set the random seed of the splitter
with the argument `random_state=0`.

In [ ]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto,
                                         test_size=196,
                                         random_state=0)

Now we can fit a linear regression using only the observations corresponding to the training set `Auto_train`.

In [ ]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

We now use the `predict()` method of `results` evaluated on the model matrix for this model
created using the validation data set. We also calculate the validation MSE of our model.

In [ ]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((y_valid - valid_pred)**2)

Hence our estimate for the validation MSE of  the linear regression
fit is $23.62$.

We can also estimate the validation error for
higher-degree polynomial regressions. We first provide a function `evalMSE()` that takes a model string as well
as training and test sets and returns the MSE on the test set.

In [ ]:
def evalMSE(terms,
            response,
            train,
            test):

   mm = MS(terms)
   X_train = mm.fit_transform(train)
   y_train = train[response]

   X_test = mm.transform(test)
   y_test = test[response]

   results = sm.OLS(y_train, X_train).fit()
   test_pred = results.predict(X_test)

   return np.mean((y_test - test_pred)**2)

Let’s use this function to estimate the validation MSE
using linear, quadratic and cubic fits. We use the `enumerate()`  function
here, which gives both the values and indices of objects as one iterates
over a for loop.

In [ ]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

These error rates are $23.62, 18.76$, and $18.80$, respectively. If we
choose a different training/validation split instead, then we
can expect somewhat different errors on the validation set.

In [ ]:
Auto_train, Auto_valid = train_test_split(Auto,
                                          test_size=196,
                                          random_state=3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

Using this split of the observations into a training set and a validation set,
we find that the validation set error rates for the models with linear, quadratic, and cubic terms are $20.76$, $16.95$, and $16.97$, respectively.

These results are consistent with our previous findings: a model that
predicts `mpg` using a quadratic function of `horsepower`
performs better than a model that involves only a linear function of
`horsepower`, and there is no evidence of an improvement in using a cubic function of `horsepower`.

> ✅ <span style="color:red">**Checkpoint** — **Validation-set approach**</span>
>
> 1. <span style="color:red">With `random_state=0` the quadratic model has a lower validation MSE than the linear one. Does the cubic term help?</span>
> 2. <span style="color:red">Why do the three MSE values change when `random_state=3` is used?</span>

## 3.2 Cross-Validation
In theory, the cross-validation estimate can be computed for any generalized
linear model.  {}
In practice, however, the simplest way to cross-validate in
Python is to use `sklearn`, which has a different interface or API
than `statsmodels`, the code we have been using to fit GLMs.

This is a problem which often confronts data scientists: "I have a function to do task $A$, and need to feed it into something that performs task $B$, so that I can compute $B(A(D))$, where $D$ is my data." When $A$ and $B$ don’t naturally speak to each other, this
requires the use of a *wrapper*.
In the `ISLP` package,
we provide
a wrapper, `sklearn_sm()`, that enables us to easily use the cross-validation tools of `sklearn` with
models fit by `statsmodels`.

The class `sklearn_sm()`
has  as its first argument
a model from `statsmodels`. It can take two additional
optional arguments: `model_str` which can be
used to specify a formula, and `model_args` which should
be a dictionary of additional arguments used when fitting
the model. For example, to fit a logistic regression model
we have to specify a `family` argument. This
is passed as `model_args={'family':sm.families.Binomial()}`.

Here is our wrapper in action:

In [ ]:
hp_model = sklearn_sm(sm.OLS,
                      MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,
                            X,
                            Y,
                            cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

The arguments to `cross_validate()` are as follows: an
object with the appropriate `fit()`, `predict()`,
and `score()` methods,  an
array of features `X` and a response `Y`.
We also included an additional argument `cv` to `cross_validate()`; specifying an integer
$k$ results in $k$-fold cross-validation. We have provided a value
corresponding to the total number of observations, which results in
leave-one-out cross-validation (LOOCV). The `cross_validate()`  function produces a dictionary with several components;
we simply want the cross-validated test score here (MSE), which is estimated to be 24.23.

We can repeat this procedure for increasingly complex polynomial fits.
To automate the process, we again
use a for loop which iteratively fits polynomial
regressions of degree 1 to 5, computes the
associated cross-validation error, and stores it in the $i$th element
of the vector `cv_error`. The variable `d` in the for loop
corresponds to the degree of the polynomial. We begin by initializing the
vector. This command may take a couple of seconds to run.

In [ ]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

As in Figure 5.4, we see a sharp drop in the estimated test MSE between the linear and
quadratic fits, but then no clear improvement from using higher-degree polynomials.

Above we introduced the `outer()`  method of the `np.power()`
function.  The `outer()` method is applied to an operation
that has two arguments, such as `add()`, `min()`, or
`power()`.
It has two arrays as
arguments, and then forms a larger
array where the operation is applied to each pair of elements of the
two arrays.

In [ ]:
A = np.array([3, 5, 9])
B = np.array([2, 4])
np.add.outer(A, B)

In the CV example above, we used $k=n$, but of course we can also use $k<n$. The code is very similar
to the above (and is significantly faster). Here we use `KFold()` to partition the data into $k=10$ random groups. We use `random_state` to set a random seed and initialize a vector `cv_error` in which we will store the CV errors corresponding to the
polynomial fits of degrees one to five.

In [ ]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0) # use same splits for each degree
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

Notice that the computation time is much shorter than that of LOOCV.
(In principle, the computation time for LOOCV for a least squares
linear model should be faster than for $k$-fold CV, due to the
availability of the formula~(5.2)  for LOOCV;
however, the generic `cross_validate()`  function does not make
use of this formula.)  We still see little evidence that using cubic
or higher-degree polynomial terms leads to a lower test error than simply
using a quadratic fit.

The `cross_validate()` function is flexible and can take
different splitting mechanisms as an argument. For instance, one can use the `ShuffleSplit()`
function to implement
the validation set approach just as easily as $k$-fold cross-validation.

In [ ]:
validation = ShuffleSplit(n_splits=1,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation);
results['test_score']

One can estimate the variability in the test error by running the following:

In [ ]:
validation = ShuffleSplit(n_splits=10,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation)
results['test_score'].mean(), results['test_score'].std()

> ✅ <span style="color:red">**Checkpoint** — **Cross-validation**</span>
>
> 1. <span style="color:red">In the call `cross_validate(hp_model, X, Y, cv=Auto.shape[0])`, why does `cv=Auto.shape[0]` give LOOCV?</span>
> 2. <span style="color:red">Compare the LOOCV and 10-fold CV errors for degrees 1–5. Which degree would you choose and why?</span>
> 3. <span style="color:red">What does `np.power.outer(H, np.arange(d+1))` produce?</span>

Note that this standard deviation is not a valid estimate of the
sampling variability of the mean test score or the individual scores, since the randomly-selected training
samples overlap and hence introduce correlations. But it does give an
idea of the Monte Carlo variation
incurred by picking different random folds.

## 3.3 The Bootstrap
We illustrate the use of the bootstrap in the simple example
 {of Section 5.2,}  as well as on an example involving
estimating the accuracy of the linear regression model on the  `Auto`
data set.
### Estimating the Accuracy of a Statistic of Interest
One of the great advantages of the bootstrap approach is that it can
be applied in almost all situations. No complicated mathematical
calculations are required. While there are several implementations
of the bootstrap in Python, its use for estimating
standard error is simple enough that we write our own function
below for the case when our data is stored
in a dataframe.

To illustrate the bootstrap, we
start with a simple example.
The  `Portfolio`  data set in the `ISLP` package is described
in Section 5.2. The goal is to estimate the
sampling variance of the parameter $\alpha$ given in formula~(5.7).  We will
create a function
`alpha_func()`, which takes as input a dataframe `D` assumed
to have columns `X` and `Y`, as well as a
vector `idx` indicating which observations should be used to
estimate
$\alpha$. The function then outputs the estimate for $\alpha$ based on
the selected observations.

In [ ]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
   cov_ = np.cov(D[['X','Y']].loc[idx], rowvar=False)
   return ((cov_[1,1] - cov_[0,1]) /
           (cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

This function returns an estimate for $\alpha$
based on applying the minimum
    variance formula (5.7) to the observations indexed by
the argument `idx`.  For instance, the following command
estimates $\alpha$ using all 100 observations.

In [ ]:
alpha_func(Portfolio, range(100))

Next we randomly select
100 observations from `range(100)`, with replacement. This is equivalent
to constructing a new bootstrap data set and recomputing $\hat{\alpha}$
based on the new data set.

In [ ]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
           rng.choice(100,
                      100,
                      replace=True))

This process can be generalized to create a simple function `boot_SE()` for
computing the bootstrap standard error for arbitrary
functions that take only a data frame as an argument.

In [ ]:
def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

Notice the use of `_` as a loop variable in `for _ in range(B)`. This is often used if the value of the counter is
unimportant and simply makes sure  the loop is executed `B` times.

Let’s use our function to evaluate the accuracy of our
estimate of $\alpha$ using $B=1{,}000$ bootstrap replications.

In [ ]:
alpha_SE = boot_SE(alpha_func,
                   Portfolio,
                   B=1000,
                   seed=0)
alpha_SE

The final output shows that the bootstrap estimate for ${\rm SE}(\hat{\alpha})$ is $0.0912$.

### Estimating the Accuracy of a Linear Regression Model
The bootstrap approach can be used to assess the variability of the
coefficient estimates and predictions from a statistical learning
method. Here we use the bootstrap approach in order to assess the
variability of the estimates for $\beta_0$ and $\beta_1$, the
intercept and slope terms for the linear regression model that uses
`horsepower` to predict `mpg` in the  `Auto`  data set. We
will compare the estimates obtained using the bootstrap to those
obtained using the formulas for ${\rm SE}(\hat{\beta}_0)$ and
${\rm SE}(\hat{\beta}_1)$ described in Section 3.1.2.

To use our `boot_SE()` function, we must write a function (its
first argument)
that takes a data frame `D` and indices `idx`
as its only arguments. But here we want to bootstrap a specific
regression model, specified by a model formula and data. We show how
to do this in a few simple steps.

We start by writing a generic
function `boot_OLS()` for bootstrapping a regression model that takes a formula to
define the corresponding regression. We use the `clone()` function to
make a copy of the formula that can be refit to the new dataframe. This means
that any derived features such as those defined by `poly()`
(which we will see shortly),
will be re-fit on the resampled data frame.

In [ ]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

This is not quite what is needed as the first argument to
`boot_SE()`. The first two arguments which specify the model will not change in the
bootstrap process, and we would like to *freeze* them.   The
function `partial()` from the `functools` module  does precisely this: it takes a function
as an argument, and freezes some of its arguments, starting from the
left. We use it to freeze the first two model-formula arguments of `boot_OLS()`.

In [ ]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

Typing `hp_func?` will show that it has two arguments `D`
and `idx` --- it is a version of `boot_OLS()` with the first
two arguments frozen --- and hence is ideal as the first argument for `boot_SE()`.

The `hp_func()` function can now be used in order to create
bootstrap estimates for the intercept and slope terms by randomly
sampling from among the observations with replacement. We first
demonstrate its utility on 10 bootstrap samples.

In [ ]:
rng = np.random.default_rng(0)
np.array([hp_func(Auto,
          rng.choice(Auto.index,
                     392,
                     replace=True)) for _ in range(10)])

Next, we use the `boot_SE()` {}  function to compute the standard
errors of 1,000 bootstrap estimates for the intercept and slope terms.

In [ ]:
hp_se = boot_SE(hp_func,
                Auto,
                B=1000,
                seed=10)
hp_se

This indicates that the bootstrap estimate for ${\rm SE}(\hat{\beta}_0)$ is
0.85, and that the bootstrap
estimate for ${\rm SE}(\hat{\beta}_1)$ is
0.0074.  As discussed in
Section 3.1.2, standard formulas can be used to compute
the standard errors for the regression coefficients in a linear
model. These can be obtained using the `summarize()`  function
from `ISLP.sm`.

In [ ]:
hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

The standard error estimates for $\hat{\beta}_0$ and $\hat{\beta}_1$
obtained using the formulas  from Section 3.1.2  are
0.717 for the
intercept and
0.006 for the
slope. Interestingly, these are somewhat different from the estimates
obtained using the bootstrap.  Does this indicate a problem with the
bootstrap? In fact, it suggests the opposite.  Recall that the
standard formulas given in
Equation 3.8 on page 75
rely on certain assumptions. For example,
they depend on the unknown parameter $\sigma^2$, the noise
variance. We then estimate $\sigma^2$ using the RSS. Now although the
formulas for the standard errors do not rely on the linear model being
correct, the estimate for $\sigma^2$ does.  We see
in Figure 3.8 on page 99  that there is
a non-linear relationship in the data, and so the residuals from a
linear fit will be inflated, and so will $\hat{\sigma}^2$.  Secondly,
the standard formulas assume (somewhat unrealistically) that the $x_i$
are fixed, and all the variability comes from the variation in the
errors $\epsilon_i$.  The bootstrap approach does not rely on any of
these assumptions, and so it is likely giving a more accurate estimate
of the standard errors of $\hat{\beta}_0$ and $\hat{\beta}_1$ than
the results from `sm.OLS`.

Below we compute the bootstrap standard error estimates and the
standard linear regression estimates that result from fitting the
quadratic model to the data. Since this model provides a good fit to
the data (Figure 3.8), there is now a better
correspondence between the bootstrap estimates and the standard
estimates of ${\rm SE}(\hat{\beta}_0)$, ${\rm SE}(\hat{\beta}_1)$ and
${\rm SE}(\hat{\beta}_2)$.

In [ ]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS,
                    quad_model,
                    'mpg')
boot_SE(quad_func, Auto, B=1000)

We  compare the results to the standard errors computed using `sm.OLS()`.

In [ ]:
M = sm.OLS(Auto['mpg'],
           quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

## <span style="color:red">3.4 Extra worked examples</span>

<span style="color:red">**Worked Example 1 – The LOOCV shortcut.** For least squares, LOOCV can be computed from a single fit using the leverages $h_i$. We check that the shortcut gives exactly the same number as the brute-force computation for the quadratic model `mpg ~ poly(horsepower, 2)`.</span>

In [ ]:
# [ADDED] worked example – run this cell
import time
quad_design = MS([poly('horsepower', 2)])
X_q = quad_design.fit_transform(Auto)
fit_q = sm.OLS(Auto['mpg'], X_q).fit()
h = fit_q.get_influence().hat_matrix_diag
loocv_shortcut = np.mean(((Auto['mpg'] - fit_q.fittedvalues) / (1 - h))**2)

t0 = time.time()
Xq_raw = np.power.outer(np.asarray(Auto['horsepower']), np.arange(3))   # columns 1, x, x^2
loocv_brute = np.mean(cross_validate(sklearn_sm(sm.OLS), Xq_raw, Auto['mpg'], cv=Auto.shape[0],
                                     scoring='neg_mean_squared_error')['test_score']) * -1
print(f"LOOCV shortcut : {loocv_shortcut:.4f}  (one fit)")
print(f"LOOCV brute    : {loocv_brute:.4f}  ({Auto.shape[0]} fits, {time.time() - t0:.1f} s)")

<span style="color:red">*Interpretation:* both numbers agree (≈ 19.25). The shortcut needs one regression instead of 392 – but it only holds for least-squares fits; for other methods (logistic regression, KNN, trees) we need the general procedure.</span>

<span style="color:red">**Worked Example 2 – A percentile bootstrap interval for the median.** There is no simple formula for the standard error of a sample median. We bootstrap the median of `mpg`.</span>

In [ ]:
# [ADDED] worked example – run this cell
from matplotlib.pyplot import subplots
rng = np.random.default_rng(171)
mpg = np.asarray(Auto['mpg'])
B = 2000
boot_medians = np.array([np.median(rng.choice(mpg, size=mpg.shape[0], replace=True)) for _ in range(B)])
lo, hi = np.percentile(boot_medians, [2.5, 97.5])
print(f"sample median = {np.median(mpg):.2f}, bootstrap SE = {boot_medians.std(ddof=1):.3f}")
print(f"95% percentile interval: [{lo:.2f}, {hi:.2f}]")
fig, ax = subplots(figsize=(6, 3.5))
ax.hist(boot_medians, bins=30, color='lightgray', edgecolor='k')
ax.axvline(lo, c='r', ls='--'); ax.axvline(hi, c='r', ls='--')
ax.set_xlabel('bootstrap median of mpg'); ax.set_ylabel('count');

<span style="color:red">*Interpretation:* the bootstrap distribution of the median is lumpy because `mpg` takes few distinct values, but its spread still gives a usable standard error and interval without any formula.</span>

# <span style="color:red">4. Exercises</span>

<span style="color:red">For every exercise write **code + output + interpretation**. Exercises 1–6 are compulsory; 7–8 are challenge (bonus) exercises.</span>

## <span style="color:red">Exercise 1 – Validation-set error for logistic regression (`Default`)</span>

- <span style="color:red">**(a)** Fit a logistic regression of `default` on `income` and `balance` using all data.</span>
- <span style="color:red">**(b)** Using the validation-set approach (half training, half validation) estimate the test error rate: split, fit on the training part, classify as `Yes` when the posterior probability exceeds 0.5, compute the validation error.</span>
- <span style="color:red">**(c)** Repeat (b) with three different splits. Comment on the variability.</span>
- <span style="color:red">**(d)** Add the dummy variable `student`. Does it lower the estimated test error?</span>

In [ ]:
# ✍️ Exercise 1 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Default = load_data('Default')
X_d = MS(['income', 'balance']).fit_transform(Default)
y_d = (Default['default'] == 'Yes').astype(int)
errors = []
for seed in [0, 1, 2]:
    pass   # TODO (b)/(c): train_test_split(..., test_size=0.5, random_state=seed); sm.GLM(...).fit(); errors.append(...)
print(errors)


## <span style="color:red">Exercise 2 – Bootstrap standard errors of logistic regression coefficients</span>

- <span style="color:red">**(a)** Report the standard errors of the `income` and `balance` coefficients given by `sm.GLM(...).fit()`.</span>
- <span style="color:red">**(b)** Write a function `boot_fn(D, idx)` that returns the coefficient estimates of the model fitted on the rows `idx` of `D`.</span>
- <span style="color:red">**(c)** Use the lab's `boot_SE()` function with $B = 1000$ to estimate the standard errors.</span>
- <span style="color:red">**(d)** Compare (a) and (c). Are the formula-based standard errors trustworthy here?</span>

In [ ]:
# ✍️ Exercise 2 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
def boot_fn(D, idx):
    D_ = D.loc[idx]
    X_ = MS(['income', 'balance']).fit_transform(D_)
    y_ = (D_['default'] == 'Yes').astype(int)
    return sm.GLM(y_, X_, family=sm.families.Binomial()).fit().params
print(boot_fn(Default, Default.index))
# TODO (c): boot_SE(boot_fn, Default, B=1000, seed=0)   (takes about 1 minute)


## <span style="color:red">Exercise 3 – LOOCV written as an explicit loop (`Weekly`)</span>

- <span style="color:red">**(a)** Fit a logistic regression of `Direction` on `Lag1` and `Lag2` using all observations, then using all but the first observation. Is the first observation classified correctly?</span>
- <span style="color:red">**(b)** Write a `for` loop over $i = 1,\dots,n$: fit the model without observation $i$, predict its direction, record an error indicator.</span>
- <span style="color:red">**(c)** Average the indicators to obtain the LOOCV error rate. Compare it with the error rate of always predicting `Up`.</span>

In [ ]:
# ✍️ Exercise 3 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Weekly = load_data('Weekly')
X_wk = MS(['Lag1', 'Lag2']).fit_transform(Weekly)
y_wk = (Weekly['Direction'] == 'Up').astype(int)
n = Weekly.shape[0]
err = np.zeros(n)
for i in range(n):
    pass   # TODO (b): keep = np.arange(n) != i ; fit on keep ; predict observation i ; err[i] = ...
print('LOOCV error rate:', err.mean())


## <span style="color:red">Exercise 4 – Cross-validation on simulated data</span>

- <span style="color:red">**(a)** Generate data with `rng = np.random.default_rng(1)`: `x = rng.normal(size=100)`, `y = x - 2 * x**2 + rng.normal(size=100)`. What are $n$ and $p$? Write the model in equation form.</span>
- <span style="color:red">**(b)** Draw a scatter plot of $y$ against $x$.</span>
- <span style="color:red">**(c)** Compute the LOOCV errors of the four models $Y = \beta_0 + \beta_1X + \dots + \beta_dX^d + \varepsilon$, $d = 1,2,3,4$.</span>
- <span style="color:red">**(d)** Repeat (c) with another seed. Are the results the same? Why?</span>
- <span style="color:red">**(e)** Which model has the smallest LOOCV error? Is this what you expected? Do the $p$-values of the fitted coefficients agree with the CV conclusion?</span>

In [ ]:
# ✍️ Exercise 4 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
rng = np.random.default_rng(1)
x4 = rng.normal(size=100)
y4 = x4 - 2 * x4**2 + rng.normal(size=100)
cv4 = np.zeros(4)
for i, d in enumerate(range(1, 5)):
    X4 = np.power.outer(x4, np.arange(d + 1))
    # TODO (c): cv4[i] = -cross_validate(sklearn_sm(sm.OLS), X4, y4, cv=100, scoring='neg_mean_squared_error')['test_score'].mean()
print(cv4)


## <span style="color:red">Exercise 5 – Bootstrap on the `Boston` housing data</span>

- <span style="color:red">**(a)** Estimate the population mean $\hat\mu$ of `medv` and its standard error $s/\sqrt{n}$.</span>
- <span style="color:red">**(b)** Estimate the standard error of $\hat\mu$ with the bootstrap. Compare with (a).</span>
- <span style="color:red">**(c)** Give a 95% confidence interval $\hat\mu \pm 2\,\mathrm{SE}$ from the bootstrap and compare it with `scipy.stats.ttest_1samp(...).confidence_interval()`.</span>
- <span style="color:red">**(d)** Estimate the median $\hat\mu_{med}$ and the 10th percentile $\hat\mu_{0.1}$ of `medv`, with bootstrap standard errors.</span>

In [ ]:
# ✍️ Exercise 5 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
Boston = load_data('Boston')
medv = np.asarray(Boston['medv'])
mu_hat = medv.mean()
se_formula = medv.std(ddof=1) / np.sqrt(len(medv))
print(mu_hat, se_formula)
# TODO (b)–(d): reuse the idea of Worked Example 2 with np.mean, np.median, np.percentile(..., 10)


## <span style="color:red">Exercise 6 – Choosing flexibility with k-fold CV (additional data: diabetes)</span>

<span style="color:red">Using `load_extra('diabetes')`, predict `progression` from `bmi` with polynomials of degree 1–8.</span>

- <span style="color:red">**(a)** Compute the 5-fold CV error, the 10-fold CV error and the LOOCV error for each degree (use `KFold(..., shuffle=True, random_state=0)`).</span>
- <span style="color:red">**(b)** Plot the three curves against the degree. Which degree would you choose? Apply the **one-standard-error rule** with the 10-fold results.</span>
- <span style="color:red">**(c)** Repeat the 10-fold CV with five different `random_state` values. How much do the curves move compared with LOOCV?</span>

In [ ]:
# ✍️ Exercise 6 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
diabetes = load_extra('diabetes')
bmi, prog = np.asarray(diabetes['bmi']), np.asarray(diabetes['progression'])
degrees = range(1, 9)
cv5, cv10, loo = [], [], []
# TODO (a): for d in degrees: X = np.power.outer(bmi, np.arange(d + 1)); ... cross_validate(sklearn_sm(sm.OLS), X, prog, cv=KFold(5, shuffle=True, random_state=0), scoring='neg_mean_squared_error')


## <span style="color:red">Challenge exercises (optional, bonus)</span>

### <span style="color:red">Exercise 7 – How many observations are in a bootstrap sample?</span>

- <span style="color:red">**(a)** Show that the probability that the $j$-th observation is **not** in a bootstrap sample of size $n$ is $(1-1/n)^n$.</span>
- <span style="color:red">**(b)** Plot $1-(1-1/n)^n$ for $n = 1,\dots,100{,}000$ (log scale). What is the limit?</span>
- <span style="color:red">**(c)** Check by simulation for $n = 100$: repeat 10,000 times `4 in rng.choice(100, 100, replace=True)` and compute the proportion of `True`.</span>

In [ ]:
# ✍️ Exercise 7 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
n_vals = np.arange(1, 100001)
# TODO (b): plot 1 - (1 - 1/n_vals)**n_vals
rng = np.random.default_rng(10)
# TODO (c): np.mean([4 in rng.choice(100, 100, replace=True) for _ in range(10000)])


### <span style="color:red">Exercise 8 – Is cross-validation accurate?</span>

<span style="color:red">Generate a large "population" from $y = \sin(2x) + 0.5x + \varepsilon$ as in Lab 1. Repeat 50 times: draw a training sample of size 100, estimate the test MSE of a degree-5 polynomial by 10-fold CV, and compute its **true** test MSE on 10,000 fresh observations. Plot CV estimate against true test MSE and comment on the bias and variance of the CV estimate.</span>

In [ ]:
# ✍️ Exercise 8 – your code
# Write your code below (replace the TODOs), then run the cell (Shift+Enter).
f8 = lambda x: np.sin(2 * x) + 0.5 * x
rng = np.random.default_rng(8)
cv_est, true_mse = [], []
# TODO: loop 50 times


# <span style="color:red">5. Submission Instructions</span>

## <span style="color:red">5.1 What to submit</span>

<span style="color:red">Submit **one link** – to a **Google Drive folder** – that contains your lab files:</span>

1. <span style="color:red">**Notebook** `IT171IU_Lab04_FullName_StudentID.ipynb` – the completed notebook with **all cells executed** (*Runtime → Run all* before saving), all ✍️ cells filled in and the written answers typed in the text cells.</span>
2. <span style="color:red">**Report** `IT171IU_Lab04_Report.pdf` (or `.docx`) – this handout with the *Student Work Areas* filled in: short answers, key numbers and the most important plots/screenshots.</span>
3. <span style="color:red">**README.md** – your full name, student ID, class/group, how to run the notebook, and a note on any AI-assistant use (allowed only as stated in the course policy).</span>


<span style="color:red">*(added)*</span>
```text
IT171IU_Lab04_FullName_StudentID/
├── IT171IU_Lab04_FullName_StudentID.ipynb
├── IT171IU_Lab04_Report.pdf
├── figures/            (optional – exported plots)
└── README.md
```

## <span style="color:red">5.2 How to share the folder</span>


<span style="color:red">**Google Drive:** upload the folder → right-click → *Share* → *General access: Anyone with the link – Viewer* → *Copy link*.</span>


## <span style="color:red">5.3 Submit the link in the Google Form</span>

| <span style="color:red">Item</span> | <span style="color:red">Details</span> |
|---|---|
| <span style="color:red">Submission form for **Lab 4** (Lab Number pre-selected)</span> | <span style="color:red">[Open the Lab 4 submission form](https://docs.google.com/forms/d/e/1FAIpQLSeM7zOP9JarqtfBxE2RpljzppuZEfMPfygg7_k96zD5VlgZfQ/viewform?usp=pp_url&entry.1104082638=Lab+4:+Resampling+Methods+(Ch.+5))</span> |
| <span style="color:red">General form link / short link</span> | <span style="color:red">[https://forms.gle/pUebamKV64bYB6KHA](https://forms.gle/pUebamKV64bYB6KHA) – choose *Lab 4: Resampling Methods (Ch. 5)*</span> |
| <span style="color:red">Fields to fill in</span> | <span style="color:red">Full Name · Student ID · Class/Group · Lab Number · Submission Link/File</span> |
| <span style="color:red">Deadline</span> | <span style="color:red">End of the lab week – Sunday 23:59 (unless the instructor announces another date)</span> |

<span style="color:red">**Before you press Submit – checklist**</span>

- <span style="color:red">The link opens in a private/incognito browser window (sharing is set correctly).</span>
- <span style="color:red">The notebook runs from top to bottom without errors (*Runtime → Restart session and run all*).</span>
- <span style="color:red">Every exercise has code **and** a short written interpretation of the result.</span>
- <span style="color:red">File and folder names follow `IT171IU_Lab04_FullName_StudentID`.</span>